In [0]:
%run ./99_Config

In [0]:
# ============================================================
# Banking Data Engineering Platform
# Notebook : 98_Utilities
# Purpose  : Common Utility Functions
# Compatible : Databricks Free Edition
# ============================================================

from pyspark.sql import DataFrame
from pyspark.sql.functions import *
from datetime import datetime

In [0]:
def print_header(title: str):

    print("=" * 80)
    print(title)
    print("=" * 80)
    print(f"Pipeline : {PIPELINE_NAME}")
    print(f"Run ID   : {CURRENT_RUN_ID}")
    print(f"Started  : {datetime.now()}")
    print("=" * 80)

In [0]:
def print_footer():

    print("=" * 80)
    print("Notebook Completed Successfully")
    print(f"Completed : {datetime.now()}")
    print("=" * 80)

In [0]:
def discover_source_systems():

    folders = sorted([

        folder.name.replace("/", "")

        for folder in dbutils.fs.ls(INCOMING_PATH)

    ])

    return folders

In [0]:
def discover_bronze_datasets():

    folders = sorted([

        folder.name.replace("/", "")

        for folder in dbutils.fs.ls(BRONZE_PATH)

    ])

    return folders

In [0]:
def discover_silver_datasets():

    folders = sorted([

        folder.name.replace("/", "")

        for folder in dbutils.fs.ls(SILVER_PATH)

    ])

    return folders

In [0]:
def read_csv(path, schema=None):

    reader = (

        spark.read

        .option("header", "true")

    )

    if schema is not None:

        reader = reader.schema(schema)

    return reader.csv(path)

In [0]:
def read_delta(path):

    return (

        spark.read

        .format("delta")

        .load(path)

    )

In [0]:
def write_delta(df, path, mode="append"):

    (

        df.write

        .format("delta")

        .mode(mode)

        .option("overwriteSchema", "true")

        .save(path)

    )

In [0]:
def create_delta_table(table_name, location):

    spark.sql(f"""
    CREATE OR REPLACE TABLE {CATALOG_NAME}.{SCHEMA_NAME}.{table_name}
    USING DELTA
    AS SELECT * FROM delta.`{location}`
    """)

In [0]:
def move_processed_files(source_folder, destination_folder):

    files = dbutils.fs.ls(source_folder)

    csv_files = [

        file

        for file in files

        if file.path.lower().endswith(".csv")

    ]

    dbutils.fs.mkdirs(destination_folder)

    for file in csv_files:

        dbutils.fs.mv(

            file.path,

            f"{destination_folder}/{file.name}"

        )

    return len(csv_files)

In [0]:
def validate_folder(path):

    try:

        dbutils.fs.ls(path)

        return True

    except:

        return False

In [0]:
def create_folder(path):

    dbutils.fs.mkdirs(path)

In [0]:
def get_csv_files(folder_path):

    files = dbutils.fs.ls(folder_path)

    return [

        file.path

        for file in files

        if file.path.lower().endswith(".csv")

    ]

In [0]:
def read_bronze(dataset):

    return read_delta(

        f"{BRONZE_PATH}/{dataset}"

    )

In [0]:
def write_silver(df, dataset):

    path = f"{SILVER_PATH}/{dataset}"

    write_delta(

        df,

        path,

        "overwrite"

    )

    create_delta_table(

        f"silver_{dataset}",

        path

    )

In [0]:
def write_rejected(df, dataset):

    path = f"{REJECTED_PATH}/{dataset}"

    write_delta(

        df,

        path,

        "overwrite"

    )

    create_delta_table(

        f"rejected_{dataset}",

        path

    )

In [0]:
def record_count(df):

    return df.count()

In [0]:
def display_summary(dataset,
                    records_read,
                    records_written,
                    rejected):

    print("="*80)

    print(f"Dataset           : {dataset}")

    print(f"Records Read      : {records_read}")

    print(f"Records Written   : {records_written}")

    print(f"Rejected Records  : {rejected}")

    print("="*80)

In [0]:
print_header("98_Utilities Loaded")

print("Available Source Systems")

print(discover_source_systems())

print_footer()